<a href="https://colab.research.google.com/github/SalvarezJ/Crib-Sentry/blob/main/notebooks/03_train_hazards.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Crib Sentry: Train the Hazard Detector
This section trains YOLO11 on the CribHD-T and CribHD-B images to find hard toys, soft toys and blankets.
The CribHD labels are outlines, so the notebook changes them to boxes and merges the two sets.
The training uses the augmentation from my Blueprint: darker lighting, blur and rotation.

In [ ]:
!pip install ultralytics albumentations

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.5/95.5 kB 4.0 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

folder = '/content/drive/MyDrive/crib_sentry_v2'
zip_path = folder + '/CribHD.zip'

os.makedirs(folder, exist_ok=True)

if not os.path.exists(zip_path):
    !wget -O {zip_path} https://coe.northeastern.edu/Research/AClab/CribHD/CribHD.zip

!unzip -q -o {zip_path} -d /content/

--2026-10-03 20:36:18--  https://coe.northeastern.edu/Research/AClab/CribHD/CribHD.zip
Resolving coe.northeastern.edu (coe.northeastern.edu)... 129.10.32.107
Connecting to coe.northeastern.edu (coe.northeastern.edu)|129.10.32.107|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 640664878 (611M) [application/zip]
Saving to: ‘/content/drive/MyDrive/crib_sentry_v2/CribHD.zip’

/content/drive/MyDr 100%[===================>] 610.99M  10.5MB/s    in 60s     

2026-10-03 20:37:20 (10.2 MB/s) - ‘/content/drive/MyDrive/crib_sentry_v2/CribHD.zip’ saved [640664878/640664878]



In [ ]:
!find /content/CribHD -maxdepth 3 -type d
!find /content/CribHD -name "*.yaml" -exec echo {} \; -exec cat {} \;
!find /content/CribHD -name "*.txt" | head -6

/content/CribHD
/content/CribHD/CribHD-B
/content/CribHD/CribHD-B/COCO
/content/CribHD/CribHD-B/COCO/train
/content/CribHD/CribHD-B/COCO/valid
/content/CribHD/CribHD-B/COCO/test
/content/CribHD/CribHD-B/PascalVOC
/content/CribHD/CribHD-B/PascalVOC/train
/content/CribHD/CribHD-B/PascalVOC/valid
/content/CribHD/CribHD-B/PascalVOC/test
/content/CribHD/CribHD-B/YoloV8
/content/CribHD/CribHD-B/YoloV8/train
/content/CribHD/CribHD-B/YoloV8/valid
/content/CribHD/CribHD-B/YoloV8/test
/content/CribHD/CribHD-C
/content/CribHD/CribHD-T
/content/CribHD/CribHD-T/COCO
/content/CribHD/CribHD-T/COCO/train
/content/CribHD/CribHD-T/COCO/valid
/content/CribHD/CribHD-T/COCO/test
/content/CribHD/CribHD-T/PascalVOC
/content/CribHD/CribHD-T/PascalVOC/train
/content/CribHD/CribHD-T/PascalVOC/valid
/content/CribHD/CribHD-T/PascalVOC/test
/content/CribHD/CribHD-T/YoloV8
/content/CribHD/CribHD-T/YoloV8/train
/content/CribHD/CribHD-T/YoloV8/valid
/content/CribHD/CribHD-T/YoloV8/test
/content/CribHD/CribHD-B/YoloV8

In [ ]:
import glob, shutil

SRC = '/content/CribHD'
DST = '/content/hazards'

# subset folder, prefix for the file name, number to add to the class ID
subsets = [
    ('CribHD-T', 'T_', 0),   # hard-toy stays 0 and soft-toy stays 1
    ('CribHD-B', 'B_', 2),   # blanket 0 becomes 2
]

if os.path.exists(DST):
    shutil.rmtree(DST)       # start clean on each run

for split in ['train', 'valid', 'test']:
    os.makedirs(DST + '/' + split + '/images')
    os.makedirs(DST + '/' + split + '/labels')

    for folder, prefix, shift in subsets:
        src_split = SRC + '/' + folder + '/YoloV8/' + split

        for label_path in sorted(glob.glob(src_split + '/labels/*.txt')):
            stem = os.path.basename(label_path)[:-4]              # for example 'train252'

            images = glob.glob(src_split + '/images/' + stem + '.*')
            if len(images) == 0:
                print('No image for', label_path)
                continue
            image_path = images[0]

            new_lines = []
            for line in open(label_path):
                parts = line.split()
                if len(parts) < 5:
                    continue
                cls = int(parts[0]) + shift
                nums = [float(p) for p in parts[1:]]

                if len(nums) == 4:
                    cx, cy, w, h = nums                           # the line is a box already
                else:
                    xs = nums[0::2]                               # each x value of the outline
                    ys = nums[1::2]                               # each y value of the outline
                    cx = (min(xs) + max(xs)) / 2
                    cy = (min(ys) + max(ys)) / 2
                    w = max(xs) - min(xs)
                    h = max(ys) - min(ys)

                new_lines.append(f'{cls} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}')

            with open(DST + '/' + split + '/labels/' + prefix + stem + '.txt', 'w') as f:
                f.write('\n'.join(new_lines))

            shutil.copy(image_path, DST + '/' + split + '/images/' + prefix + os.path.basename(image_path))

print('Done')

Done


In [ ]:
for split in ['train', 'valid', 'test']:
    source_t = len(glob.glob(SRC + '/CribHD-T/YoloV8/' + split + '/labels/*.txt'))
    source_b = len(glob.glob(SRC + '/CribHD-B/YoloV8/' + split + '/labels/*.txt'))

    labels = glob.glob(DST + '/' + split + '/labels/*.txt')
    images = glob.glob(DST + '/' + split + '/images/*')

    boxes = 0
    for path in labels:
        for line in open(path):
            if line.strip():
                boxes += 1

    print(split, '| source T + B:', source_t, '+', source_b, '=', source_t + source_b,
          '| merged labels:', len(labels), '| merged images:', len(images), '| boxes:', boxes)

train | source T + B: 920 + 449 = 1369 | merged labels: 1369 | merged images: 1369 | boxes: 2783
valid | source T + B: 50 + 40 = 90 | merged labels: 90 | merged images: 90 | boxes: 153
test | source T + B: 30 + 10 = 40 | merged labels: 40 | merged images: 40 | boxes: 88


In [ ]:
yaml_text = """path: /content/hazards
train: train/images
val: valid/images
test: test/images

nc: 3
names: ['hard-toy', 'soft-toy', 'blanket']
"""

with open(DST + '/data.yaml', 'w') as f:
    f.write(yaml_text)

print(yaml_text)

path: /content/hazards
train: train/images
val: valid/images
test: test/images

nc: 3
names: ['hard-toy', 'soft-toy', 'blanket']



In [ ]:
import albumentations as A
from ultralytics import YOLO

model = YOLO('yolo11n.pt')

results = model.train(
    data=DST + '/data.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    hsv_v=0.6,                                       # darker and lighter images
    degrees=15,                                      # rotate each image up to 15 degrees
    augmentations=[A.Blur(blur_limit=7, p=0.3)],     # blur 3 of each 10 images
    project='/content/drive/MyDrive/crib_sentry_v2',
    name='hazards'
)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics 8.4.172 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, augmentations=[{'__version__': '2.0.8', 'transform': {'__class_fullname__': 'Blur', 'p': 0.3, 'blur_limit': (3, 7)}}], auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/hazards/data.yaml, degrees=15, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=Fa

## Results on the validation images
- mAP50 for all classes is 0.794.
- Hard-toy recall is 0.733 and soft-toy recall is 0.754.
- Blanket recall is 0.658, which is the lowest score.
- The scores still went up at epoch 50.

These are validation scores, so they grade the boxes and not the alerts.
The validation images are CribHD images with no crib and no baby in them.
I measure the alert accuracy on the test images in a later notebook.